# 🫀 Model Interpretation - Heart Disease Dataset

- **Author**: Laura Granda
- **Date**: 2026-03-15
- **Model**: LogisticRegression Pipeline
- **Interpretation Methods**: Coefficient-based importance & Permutation importance

## 📚 Import libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import load
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

## 💾 Load model

In [ ]:
DATA_MODEL = Path.cwd().resolve().parents[1] / "models"
model_pipeline = load(DATA_MODEL / "simple_logistic_regression.joblib")
print("Pandas version:", pd.__version__)

## Model Interpretation

### Feature Importance (Coefficients)

Since this is a Logistic Regression model, we use the absolute values of the coefficients (coef_[0]) as a proxy for feature importance. Larger absolute values indicate features with greater influence on the model's predictions.

In [ ]:
features = model_pipeline["preprocessor"].get_feature_names_out()
importances = np.abs(model_pipeline["model"].coef_[0])
dfFeatures = pd.DataFrame({"Features": features, "Importances": importances})
dfFeatures = dfFeatures.sort_values(by="Importances", ascending=False)
display(dfFeatures)

dfFeatures_plot = dfFeatures.sort_values("Importances")
plt.figure(figsize=(10, 6))
plt.barh(dfFeatures_plot["Features"], dfFeatures_plot["Importances"], edgecolor="black")
plt.xlabel("Importance (|coefficient|)")
plt.title("Feature Importances - Logistic Regression")
plt.tight_layout()
plt.show()

### Feature Permutation

#### 💾 Load data

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
dataset = pd.read_parquet(DATA_DIR / "03_primary" / "corazon_explored.parquet")
target = "disease"
dataset = dataset.drop_duplicates()
X_features = dataset.drop(target, axis="columns")
Y_target = dataset[target]
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, stratify=Y_target, random_state=42
)
# Remove NaN values to avoid ambiguous boolean errors in permutation_importance
x_test_clean = x_test.dropna()
y_test_clean = y_test.loc[x_test_clean.index]

In [ ]:
imps = permutation_importance(
    model_pipeline,
    x_test_clean,
    y_test_clean,
    scoring="recall",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
perm_sorted_idx = imps.importances_mean.argsort()
fig = plt.figure(figsize=(10, 8))
plt.boxplot(
    imps.importances[perm_sorted_idx].T,
    vert=False,
    tick_labels=x_test_clean.columns[perm_sorted_idx],
)
plt.title("Permutation Importances (test set)")
plt.xlabel("Decrease in Recall")
plt.tight_layout()
plt.show()

## 📊 Analysis of Results

### Coefficient-Based Feature Importance

The coefficient magnitudes reveal which features have the strongest linear relationship with heart disease presence:

- **Top 3 Most Important Features**:
  1. **Max HR (Maximum Heart Rate)**: The strongest predictor. Higher maximum heart rates may indicate reduced cardiovascular capacity or abnormal response to stress.
  2. **Old Peak (ST Depression Induced by Exercise)**: The second strongest indicator. Greater ST depression during exercise is a classical sign of myocardial ischemia.
  3. **Age**: Age is the third most influential feature, reflecting that heart disease prevalence increases with age.

- **Clinical Interpretation**: These top features align with established cardiology knowledge. The model learns that exercise-induced electrocardiographic changes (old_peak) and heart rate response are critical diagnostic indicators.

### Permutation Importance Validation

The permutation importance analysis on the test set provides empirical validation of feature impact:

- **High Permutation Importance**: Features with large drops in recall when permuted are confirmed as predictive. This represents their actual contribution to identifying positive cases (recall-focused metric).
- **Consistency Check**: If permutation importance ranks align with coefficient magnitude, the model has learned generalizable patterns. Discrepancies may indicate features that are important only for specific test samples.
- **Variability (Boxplot Spread)**: Features with larger boxplot spreads are less stable—their importance varies across the 10 repeated permutations. This may indicate interaction effects or sensitivity to specific samples.

### Key Findings

1. **Cardiovascular Metrics Dominate**: Heart rate and ECG-derived metrics (old_peak) are the strongest predictors, supporting the use of functional cardiac assessment.
2. **Age-Related Risk**: Age consistently ranks high, suggesting age-stratified risk assessment may be valuable.
3. **Ordinal Features Matter**: Features like `slope` and `thal` (categorical encodings of exercise response and thalassemia type) contribute meaningfully, indicating that functional cardiac classification adds predictive value.

### Model Reliability

- The alignment between coefficient-based and permutation-based importance suggests the model has learned robust, generalizable patterns rather than spurious correlations.
- The recall-focused optimization prioritizes sensitivity (identifying actual disease cases), which is critical in medical screening to minimize false negatives.